# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template and code examples for loading, exploring, and visualizing the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. This section demonstrates loading a Croissant schema and inspecting the dataset metadata.  

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset title: {getattr(metadata, 'name', '')}\nDescription: {getattr(metadata, 'description', '')}")
print(f"Published: {getattr(metadata, 'datePublished', '')}")
print(f"License: {getattr(metadata, 'license', '')}")


## 2. Data Overview
Review available record sets, their field `@id`s and other attributes. Here, we examine how the dataset is organized and find out what record sets and fields are available using their `@id` values as per Croissant.

Note: All entity references use their `@id` fields for clarity and reproducibility.

In [ ]:
from pprint import pprint

# List all record sets by their @id
record_sets = list(dataset.record_sets)
print(f"Available Record Sets ({len(record_sets)}):\n")
for rs in record_sets:
    print(f"- {rs['@id']} | name: {rs.get('name', '')}")
    # List the @id and name of each field in this RecordSet
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("    Fields:")
    for fld in fields:
        fld_id = fld['@id'] if isinstance(fld, dict) else fld
        fld_obj = dataset.field(fld_id)
        print(f"    - {fld_id} | name: {fld_obj.get('name', '') if fld_obj else ''}")
    print()


## 3. Data Extraction
Load data from a chosen record set into a DataFrame. We will reference record set and field `@id`s found above.

**Important**: Replace `<record_set_id>` and `<field_id>` below with the actual `@id` values from the previous cell for the desired RecordSet.

In [ ]:
# List all record set @ids
all_record_set_ids = [rs['@id'] for rs in record_sets]
print('All available record_set @ids:')
pprint(all_record_set_ids)

# For demonstration, choose the first available record set
chosen_record_set_id = all_record_set_ids[0]
print(f"Chosen Record Set: {chosen_record_set_id}\n")

# Extract all records into a DataFrame
records = list(dataset.records(record_set=chosen_record_set_id))
df = pd.DataFrame(records)
print(f"Fields (@id as column names):\n{df.columns.tolist()}")
print("\nPreview:")
df.head()

## 4. Exploratory Data Analysis (EDA)
We select numeric and categorical fields for basic exploration:
- Filter records based on a value threshold for a chosen numeric field
- Normalize the field
- Group by a categorical field

**Important:** 
1. Set the variables below (`numeric_field_id`, `group_field_id`) to the actual `@id` strings of the numeric/categorical fields you wish to analyze (pick from `df.columns` above).
2. You can rerun the notebook with different field choices for more exploration.

In [ ]:
# Example: choose a numeric field and a group field @id
# Replace these values with actual @id strings from df.columns as appropriate
numeric_field_id = df.columns[0]   # e.g., '@id': 'accession' (if a numeric field is available)
group_field_id = df.columns[1]     # e.g., '@id': 'description'

print(f"Numeric field candidate: {numeric_field_id}")
print(f"Group field candidate: {group_field_id}")

if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    threshold = df[numeric_field_id].mean()  # Example threshold at mean
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:\n")
    print(filtered_df.head())

    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"\nGrouped data by {group_field_id}:")
        print(grouped_df.head())
else:
    print(f"The field '{numeric_field_id}' does not appear to be numeric. Please select an appropriate numeric field @id from df.columns.")

## 5. Visualization
Below, we visualize data distributions and relationships using matplotlib, referencing fields by their `@id` as columns in the DataFrame. You can adapt the fields or plot types for further exploration.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Only plot if the chosen field is numeric
if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id], bins=30, kde=True)
    plt.title(f"Histogram of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
    
    # Boxplot for group field if it is categorical and not too many categories
    if group_field_id in df.columns and df[group_field_id].nunique() < 30:
        plt.figure(figsize=(10,5))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"Boxplot of {numeric_field_id} grouped by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()


## 6. Conclusion
- In this notebook, we loaded the FAIR² dataset's Croissant schema and explored its available record sets and fields, referencing entities by their `@id`.
- We demonstrated how to extract records as pandas DataFrames and perform basic EDA using explicit field IDs.
- You can use this template and the field `@id` references to perform further analyses, tailored to your research or data processing requirements.